## Transform Drivers Data
1. Read bronze drivers table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (driverId → driver_id, dateOfbirth → date_of_birth)
4. Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case
5. Remove duplicate records
6. Transform values of column nationality to Title Case
7. Write the transformed data to silver drivers table

In [0]:
# 1. Get batch ID
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/03_silver_helpers"

In [0]:
from pyspark.sql import functions as f

In [0]:
bronze_path = "s3://bizf1project123/Bronze/Kalashri_increment/drivers"

silver_path = "s3://bizf1project123/Silver/Kalashri_increment/drivers"

## Step 1 - Read bronze drivers table

In [0]:
drivers_df = (
    spark.read.format('delta').load(bronze_path)
         .filter((F.col("batch_id") == v_batch_id))
)

display(drivers_df)

## Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
drivers_dropped_df = drivers_df.drop(F.col("url"))

display(drivers_dropped_df)

## Step 3 - Standardise Column Names

In [0]:
drivers_renamed_df = (
    drivers_dropped_df
        .withColumnsRenamed({
            "driverId": "driver_id",
            "dateOfBirth": "date_of_birth"
        })
)

display(drivers_renamed_df)

## Step 4 - Concatenate name.givenName and name.familyName to create a new column called driver_name

In [0]:
drivers_concatenated_df = (
  drivers_renamed_df
       .withColumn("driver_name", 
                   F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
       .drop("name")
)

display(drivers_concatenated_df)

## Step 5 - Remove duplicate records


In [0]:
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])

display(drivers_distinct_df)

## Step 6 - Transform values of column nationality to Title Case

In [0]:
drivers_final_df = (
    drivers_distinct_df
        .withColumn('nationality', F.initcap(F.col("nationality")))
)

display(drivers_final_df)


## Step 7 - Write the transformed data to silver constructors table

In [0]:
write_to_silver(
    input_df=drivers_final_df,
    target_path=silver_path,
    merge_condition="t.driver_id = s.driver_id",
    columns_to_update=[
        "driver_name",
        "date_of_birth",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Silver/Kalashri_increment/drivers`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Silver/Kalashri_increment/drivers",
#     True
# )